# Transformers and Tap Changers

----

The 61970 Wires package defines several classes for representing the different components of transformers. The first is PowerTransformer, which represents the electrical network representation of a transformer for both balanced and unbalanced circuits. The second is TransformerTank, which refers to the assembly of two or more windings placed inside a tank and can be used to model single-phase and three-phase transformers. TransformerEnd is the conducting connection point of a transformer and corresponds to the terminal of a particular winding.  As with all ConductingEquipment, the PowerTransformer is connected through a set of Terminal objects, with the particular winding indicated by the TransformerEnd.endNumber attribute. The highest voltage winding should have an endNumber of 1. The endNumber does not need to match the ACDCTerminal.sequenceNumber attribute of the Terminal to which the transformer is connected. 

The BaseVoltage and Terminal are associated with the TransformerEnd of each winding, rather than with the PowerTransformer itself. Figure 16 shows the associations between the different classes used to specify transformer parameters and windings.

PowerTransformer objects may be modeled with or without specifying TransformerTank objects. In both cases the PowerTransformer.vectorGroup attribute for protective relaying should be specified according to IEC transformer standards (e.g., Dy1 for many substation transformers). 


The case without specifying TransformerTank objects is most suitable for balanced three-phase transformers that will not reference any reusable asset catalog data. This approach is typically used for transmission system modeling, where each transformer is unique. Each winding will have a PowerTransformerEnd that associates to both a Terminal and a BaseVoltage, and the parent PowerTransformer. The impedance and admittance parameters are defined by reverse-associated TransformerMeshImpedance between each pair of windings, and a reverse-associated TransformerCoreAdmittance for one winding. The units for these are ohms and siemens based on the winding voltage, rather than per-unit. WindingConnection is similar to PhaseShuntConnectionKind, adding Z and Zn for zig-zag connections and A for autotranformers. TransformerStarImpedance is used for conversion of three-winding transformers to separate two-winding equivalents, which is common practice in numerous power flow solvers.

If the transformer is unbalanced in any way, then TransformerTankEnd is used instead of PowerTransformerEnd, and then one or more TransformerTank objects may be used in the parent PowerTransformer. Some of the use cases are 1) center-tapped secondary, 2) open-delta and 3) EHV transformer banks. Tank-level modeling is also required if using catalog data to specify physical equipment ratings, etc. through the AssetInfo package. 



In [1]:
from cimgraph import utils
from mermaid import Mermaid
import cimgraph.data_profile.cim17v40 as cim

In [9]:
diagram_text = utils.get_mermaid([cim.TransformerEnd, cim.Terminal, cim.ConductingEquipment, cim.PowerTransformer, cim.BaseVoltage, cim.PowerTransformerEnd, cim.TransformerTank, cim.TransformerTankEnd, cim.TransformerStarImpedance, cim.TransformerMeshImpedance,cim.TransformerCoreAdmittance, cim.PhaseCode, cim.WindingConnection])
Mermaid(diagram_text)

Many distribution software packages use the concept of catalog data, aka library data, especially for lines and transformers. This concept is implemented in CIM through the ability to define a single set of class definitions using the IEC 61968 AssetInfo package to save a large amount of space when defining customer secondary transformers (which typically comprise hundreds or thousands of identical poletop and pad-mounted installations). A particular transformer design and rating is then defined by creating PowerTransformerInfo and TransformerTankInfo library objects that are associated with each type of specification objects. 

The rated voltage, rated current, and resistance of each winding are defined as attributes of a TransformerEndInfo object that is created for each transformer winding. It is important that the TransformerEndInfo.endNumber of the physical asset match the TransformerEnd.endNumber of its representation in the electrical circuit. The shunt admittances are defined by NoLoadTest on a winding / end, with usually just one such test. The impedances are defined by a set of attributes of ShortCircuitTest; one winding / end will be energized, and one or more of the others will be grounded in these tests. The complete list of asset properties is summarized in Figure 17. Note that these classes are associated with TransformerTankInfo (not PowerTransformerInfo) because transformer testing is done on tanks.

In [13]:
diagram_text = utils.get_mermaid([cim.TransformerTankInfo, cim.PowerTransformerInfo, cim.TransformerEndInfo, cim.OpenCircuitTest, cim.ShortCircuitTest, cim.NoLoadTest, cim.TransformerTest, cim.WindingConnection])
Mermaid(diagram_text)

The TapChanger class is used to model both phase-shifting transformers and voltage regulators through the PhaseTapChanger and RatioTapChanger classes, which are associated with the particular TransformerEnd. The highest, lowest, and neutral tap positions available are specified as positive integers such that a TapChanger with 16 tap positions would have attributes of lowStep set to 0, neutralStep set to 8, and highStep set to 16. If a particular application uses a range of -8 to 8 for the same transformer tap range, it is the responsibility of application to convert the tap ranges to the format used internally. 

If a voltage regulator uses line drop compensation, then those parameters will be defined as attributes of the TapChangerControl class, which inherits from RegulatingControl. RegulatingControl is a higher-level class that is used to specify the control mode and setpoints for numerous types of RegulatingCondEq, such capacitors, reactors, SVC, and generator automatic voltage regulation controls. Whether a particular device is regulating voltage, activePower, reactivePower, etc. is specified by the RegulatingControl:mode attribute. Other control settings, such as targetValue, targetDeadband, etc.  are also attributes of RegulatingControl, as shown in Figure 18 below.

In summary, a single-phase line voltage regulator modeled in CIM includes a PowerTransformer, a TransformerTank, a TransformerTankEnd, a RatioTapChanger, and a TapChangerControl. The CT and PT parameters of a voltage regulator can only be described via the AssetInfo mechanism, described below. The RegulationControl.mode must be voltage. Older CIM versions used the tculControlMode attribute, which is now deprecated. 

The AssetInfo package in the 61968 package defines the TapChangerInfo class with a set of attributes ctRating, ctRatio, and ptRatio needed for line drop compensator settings in voltage regulators. Catalog data is a one-to-many relationship. In this case, many TapChangers can share the same TapChangerInfo data, which saves space and provides consistency. Older versions of CIM had many-to-many catalog relationships, but now only one AssetDataSheet may be associated per Equipment.


In [19]:
diagram_text = utils.get_mermaid([cim.PowerTransformerEnd, cim.TransformerTankEnd, cim.TransformerEnd, cim.RatioTapChanger,cim.TapChanger, cim.TapChangerControl, cim.RegulatingControl, cim.RegulatingControlModeKind, cim.PhaseCode])
Mermaid(diagram_text)